<a href="https://colab.research.google.com/github/uyx7uy/ds2002-fa26/blob/main/2026-09-25%20-%20Cleaning%20Gauntlet%20-%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
before = len(df)

df = df.drop_duplicates()

log('drop duplicates', 'removed exact duplicate rows', before - len(df))
df.shape

[drop duplicates] removed exact duplicate rows (15 row(s))


(300, 5)

### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].astype(str)
df['price'] = df['price'].str.replace('$', '', regex=False)
df['price'] = df['price'].astype(float)

log('clean price', 'changed price from text to number', len(df))
df['price'].head()

[clean price] changed price from text to number (300 row(s))


,price
0,12.0
1,7.5
2,7.5
3,7.5
4,7.5


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
before = len(df)

df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
df = df.dropna(subset=['qty'])
df = df[df['qty'] >= 1]

log('clean qty', 'removed missing and negative quantities', before - len(df))
df.shape

[clean qty] removed missing and negative quantities (25 row(s))


(275, 5)

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

df['item'] = df['item'].map(ITEM_MAP)

log('clean item', 'combined 6 item spellings into 3 item names', 3)
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[clean item] combined 6 item spellings into 3 item names (3 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

df['category'] = df['category'].map(CATEGORY_MAP)

log('clean category', 'combined category names and grouped Apparel with Merch', 3)
print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[clean category] combined category names and grouped Apparel with Merch (3 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()) == {'Food', 'Merch', 'RainGear'}

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = df.groupby('category')['revenue'].sum()
revenue_by_category = revenue_by_category.sort_values(ascending=False)
revenue_by_category = revenue_by_category.round(2)

print(revenue_by_category)
print('Total revenue:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Total revenue: 4740.0


**What I would tell the vendor:** I would tell the vendor to stock more Food items because Food had the highest total revenue after cleaning.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop duplicates,removed exact duplicate rows,15
1,clean price,changed price from text to number,300
2,clean qty,removed missing and negative quantities,25
3,clean item,combined 6 item spellings into 3 item names,3
4,clean category,combined category names and grouped Apparel wi...,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.


a) The duplicate removal step changed the revenue total the most. Before dropping duplicates, the revenue total was 4852.50. After dropping duplicates, the revenue total was 4594.50.

b) A reasonable person could have kept Apparel as its own category instead of grouping it with Merch. Keeping Apparel separate would not change the overall total revenue of 4740.00, but it would split the category revenue differently. I grouped Apparel with Merch because apparel is a type of merchandise, so this made the final categories simpler.